<a href="https://colab.research.google.com/github/VGoma23/data-520-asean/blob/main/copy_decision_tree_make_rslearn_great_again.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Acknowledgements

this is all based on Mikey's existing code in the Github repo for DATA520 ASEAN project.

#Imports

In [10]:
pip install rasterio

In [11]:
import glob
from pathlib import Path
import json
import rasterio
from rasterio.crs import CRS
from rasterio import warp

In [12]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='iuu-fishing-detections-asean')

In [13]:
from google.colab import drive
drive.mount('/content/drive')

mikey = '/content/drive/MyDrive/Data_520_Project/data/mikey/mikey'
austin = '/content/drive/MyDrive/Data_520_Project/data/austin/austin'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Code

In [14]:
bands = ['B12', 'B11', 'B8A', 'B8', 'B7', 'B6', 'B5', 'B4', 'B3', 'B2'] # 5 6 7 8A 11 12
vis_bands = ['B4', 'B3', 'B2']
label = 'vessel'
test_scale = 10

In [17]:
image_collection = []
# master_image = ee.ImageCollection([])

features_collection = ee.FeatureCollection([])

for img_folder in Path(f"{mikey}").glob("*"):
  metadata = json.load( open(str(img_folder / "metadata.json")))
  img_name = open(str(img_folder / "image_name_from_siv.txt")).read()[:-5]
  dateRangeStart = ee.Date(metadata['time_range'][0])
  dateRangeEnd = ee.Date(metadata['time_range'][1])

  rectangle_bounds = ee.Geometry.Rectangle(
    [metadata['bounds'][0], metadata['bounds'][1],
     metadata['bounds'][2], metadata['bounds'][3]],
    ee.Projection(metadata['projection']['crs'],
     [10, 0, 0, 0, -10, 0])
    , True, False
    )

  image_c = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    .filterDate(dateRangeStart, dateRangeEnd)
    .filter(ee.Filter.stringContains('PRODUCT_ID',img_name))
    .select(bands, bands)
  )

  if (image_c.size().getInfo() == 0):
    continue
  image = image_c.first().clip(rectangle_bounds)


  # would put lat/lon stuff here for labeled points... will get that soon
  features = json.load( open(str(img_folder / "layers/label/data.geojson")))['features']
  features_collection_unmapped = []
  for feature in features:
    features_collection_unmapped.append(ee.Feature(ee.Geometry.Point(
    feature['geometry']['coordinates'],
    rectangle_bounds.projection()),
      {label: 1}
    ))
  # features_collection.append(image.select(bands).sampleRegions(
  #   collection=features_collection_unmapped, properties=[label], scale=10
  # ))
  # also here would create training non-vessel points for the unmarked pics
  # TODO: also get non-vessel points from images with vessels in them
  print(len(features))
  if len(features) == 0:
    non_vessel_points_unlabeled = image.select(bands).sample(
      region= image.geometry(),
        scale = test_scale,
        numPixels = 20,
        geometries= False,  # Set this to False to ignore geometries
    )
  else:
    non_vessel_points_unlabeled
  def set_label_0(feature):
    return feature.set(label, 0)
  non_vessel_points = non_vessel_points_unlabeled.map(set_label_0)
  # sample x amount of points, add them to features_collection with vessel = 0
  features_collection_unmapped_fc = ee.FeatureCollection(features_collection_unmapped)
  features_collection = features_collection.merge(image.select(bands).sampleRegions(
    collection=features_collection_unmapped_fc, properties=[label], scale=test_scale
  ))

  features_collection = features_collection.merge(non_vessel_points)

  image_collection.append(image)

master_image = ee.ImageCollection(image_collection)
# master_image

# training_features = ee.FeatureCollection(features_collection)
# training_features
features_collection

2


NameError: name 'non_vessel_points_unlabeled' is not defined

In [ ]:
Map = geemap.Map()

vis_params = {"min": 0, "max": 3000, "bands": vis_bands}

Map.addLayer(master_image, vis_params, "combined images example")

# Map.addLayer(training_features)